# Activité Pratique N°4 — Architecture Transformer
## Partie 2 — Exercice 1 : Consommation électrique avec un Transformer

**IA Générative Multi Modale — 5IAD G1, EMSI** · Étudiant : BENHAYOUN SADAFI Mohamed

Utiliser les 24 dernières heures de consommation pour prédire celle de l'heure suivante.

**Dataset :** *Individual Household Electric Power Consumption* (UCI), variable `Global_active_power` (kW), agrégée en **puissance moyenne horaire**.
Le fichier `household_power_consumption.txt` (133 Mo, 2 millions de mesures minute par minute) est lu **directement dans le zip** fourni avec le cours, dans `Supports de cours` : pas besoin de le décompresser.

In [ ]:
import copy
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


def fixer_graine(graine=42):
    random.seed(graine)
    np.random.seed(graine)
    torch.manual_seed(graine)


def nb_parametres(modele):
    return sum(p.numel() for p in modele.parameters())


fixer_graine()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| appareil :", device)

## 1. Préparation des données

In [ ]:
ZIP = Path("../Supports de cours/Activité Pratique N°4 - Architecture Transformer/"
           "individual+household+electric+power+consumption.zip")

debut = time.time()
df = pd.read_csv(ZIP, sep=";", na_values=["?"], low_memory=False, usecols=["Date", "Time", "Global_active_power"])
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%d/%m/%Y %H:%M:%S")
print(f"{len(df):,} mesures lues en {time.time() - debut:.0f} s")
print("Période :", df["datetime"].min(), "->", df["datetime"].max())
print("Valeurs manquantes :", df["Global_active_power"].isna().sum())

puissance = pd.to_numeric(df["Global_active_power"], errors="coerce")
puissance.index = pd.DatetimeIndex(df["datetime"])

# Puissance moyenne horaire : on garde les heures avec au moins 45 minutes mesurées.
horaire = puissance.resample("h").agg(["mean", "count"])
serie = horaire["mean"].where(horaire["count"] >= 45)
print(f"\n{len(serie):,} heures, dont {serie.isna().sum()} écartées (trop de mesures manquantes)")

# Découpage chronologique : 70 % entraînement, 15 % validation, 15 % test.
n = len(serie)
fin_train = int(0.70 * n)
fin_val = int(0.85 * n)
print("Entraînement jusqu'au", serie.index[fin_train - 1], "| validation jusqu'au", serie.index[fin_val - 1])

# Normalisation calculée uniquement sur l'entraînement.
moyenne = serie.iloc[:fin_train].mean()
ecart_type = serie.iloc[:fin_train].std()
valeurs = ((serie - moyenne) / ecart_type).to_numpy()
print(f"Moyenne (train) = {moyenne:.3f} kW | écart-type (train) = {ecart_type:.3f} kW")

In [ ]:
def preparer_fenetres(fenetre):
    # Fenêtres de `fenetre` heures consécutives -> heure suivante, sans valeur manquante.
    X, y, cibles = [], [], []
    for i in range(fenetre, n):
        sequence, cible = valeurs[i - fenetre:i], valeurs[i]
        if np.isfinite(sequence).all() and np.isfinite(cible):
            X.append(sequence)
            y.append(cible)
            cibles.append(i)
    X = torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1)
    y = torch.tensor(np.array(y), dtype=torch.float32).unsqueeze(-1)
    cibles = np.array(cibles)
    masques = {"train": cibles < fin_train,
               "val": (cibles >= fin_train) & (cibles < fin_val),
               "test": cibles >= fin_val}
    return X, y, cibles, masques


X24, y24, cibles24, masques24 = preparer_fenetres(24)
for nom, m in masques24.items():
    print(f"{nom:5s} : {m.sum():6d} fenêtres")

In [ ]:
plt.figure(figsize=(14, 3.5))
extrait = serie.iloc[fin_val:fin_val + 24 * 14]
plt.plot(extrait.index, extrait.values)
plt.title("Puissance active moyenne horaire — deux semaines au début du test")
plt.ylabel("kW")
plt.grid(alpha=0.3)
plt.show()

## 2. Implémentation : `EnergyTransformer`

| Élément | Configuration |
|---|---|
| Projection | `Linear(1, 32)` |
| Positions | encodage positionnel appris des 24 heures (`Embedding(24, 32)`) |
| Encodeur | 2 × `TransformerEncoderLayer(d_model=32, nhead=4, dim_feedforward=128, dropout=0.1, batch_first=True)` |
| Sortie | `Linear(32, 1)` sur la représentation de la **dernière heure** |

In [ ]:
class EnergyTransformer(nn.Module):
    def __init__(self, fenetre=24, d_model=32, nhead=4, dim_ff=128, couches=2, dropout=0.1):
        super().__init__()
        self.projection = nn.Linear(1, d_model)
        self.positions = nn.Embedding(fenetre, d_model)
        couche = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
                                            dropout=dropout, batch_first=True)
        self.encodeur = nn.TransformerEncoder(couche, num_layers=couches, enable_nested_tensor=False)
        self.sortie = nn.Linear(d_model, 1)

    def forward(self, x):
        indices = torch.arange(x.size(1), device=x.device)
        z = self.projection(x) + self.positions(indices)
        z = self.encodeur(z)
        return self.sortie(z[:, -1, :])


m = EnergyTransformer()
xb = X24[:8]
z = m.projection(xb) + m.positions(torch.arange(24))
print("Entrée                    :", tuple(xb.shape))
print("Après projection          :", tuple(z.shape))
print("Après Transformer Encoder :", tuple(m.encodeur(z).shape))
print("Prédiction                :", tuple(m(xb).shape))
print("Paramètres                :", nb_parametres(m))

## 3. Entraînement et évaluation

Adam (`lr = 0.001`), perte MSE, **20 époques au maximum**, et on **garde le modèle qui a la meilleure perte de validation**.

In [ ]:
def entrainer_energie(X, y, masques, fenetre, epochs=20, lr=1e-3, batch=64):
    fixer_graine()
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X[masques["train"]], y[masques["train"]]),
                                         batch_size=batch, shuffle=True)
    Xv, yv = X[masques["val"]].to(device), y[masques["val"]].to(device)
    modele = EnergyTransformer(fenetre=fenetre).to(device)
    optimiseur = torch.optim.Adam(modele.parameters(), lr=lr)
    critere = nn.MSELoss()
    meilleure, meilleur_etat, historique = float("inf"), None, []
    for epoch in range(1, epochs + 1):
        modele.train()
        total = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            perte = critere(modele(xb), yb)
            optimiseur.zero_grad()
            perte.backward()
            optimiseur.step()
            total += perte.item() * len(xb)
        modele.eval()
        with torch.no_grad():
            perte_val = critere(modele(Xv), yv).item()
        historique.append((total / masques["train"].sum(), perte_val))
        marque = ""
        if perte_val < meilleure:
            meilleure, meilleur_etat, marque = perte_val, copy.deepcopy(modele.state_dict()), "  <- meilleur"
        print(f"Époque {epoch:2d} | train {historique[-1][0]:.4f} | validation {perte_val:.4f}{marque}")
    modele.load_state_dict(meilleur_etat)
    return modele.eval(), historique


def predire(modele, X, batch=1024):
    with torch.no_grad():
        return torch.cat([modele(X[i:i + batch].to(device)).cpu() for i in range(0, len(X), batch)])


def en_kw(t):
    return t.numpy().ravel() * ecart_type + moyenne


debut = time.time()
modele24, histo24 = entrainer_energie(X24, y24, masques24, fenetre=24)
print(f"\nEntraînement terminé en {time.time() - debut:.0f} s")
plt.figure(figsize=(8, 3.5))
plt.plot([h[0] for h in histo24], label="train")
plt.plot([h[1] for h in histo24], label="validation")
plt.xlabel("Époque")
plt.ylabel("MSE (données normalisées)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Évaluation en kW, heure par heure** (chaque prédiction utilise les 24 mesures réellement observées avant son heure cible), comparée à deux références :
- **Persistance** : la valeur de la dernière heure observée ;
- **Référence journalière** : la valeur observée 24 heures avant l'heure cible (première valeur de la fenêtre).

In [ ]:
def mae_rmse(reel, pred):
    erreur = pred - reel
    return np.abs(erreur).mean(), np.sqrt((erreur ** 2).mean())


test = masques24["test"]
reel = en_kw(y24[test])
predictions_kw = {
    "Transformer": en_kw(predire(modele24, X24[test])),
    "Persistance (h-1)": en_kw(X24[test][:, -1, :]),
    "Référence journalière (h-24)": en_kw(X24[test][:, 0, :]),
}
tableau = pd.DataFrame({nom: dict(zip(["MAE (kW)", "RMSE (kW)"], mae_rmse(reel, p)))
                        for nom, p in predictions_kw.items()}).T
print(f"Test : {test.sum()} heures\n")
print(tableau.round(4).to_string())

In [ ]:
# Sept jours consécutifs du test (168 heures cibles qui se suivent sans trou)
cibles_test = cibles24[test]
debut_bloc = next(k for k in range(len(cibles_test) - 168) if cibles_test[k + 167] - cibles_test[k] == 167)
bloc = slice(debut_bloc, debut_bloc + 168)
dates = serie.index[cibles_test[bloc]]

plt.figure(figsize=(15, 4.5))
plt.plot(dates, reel[bloc], color="black", linewidth=2, label="Observé")
plt.plot(dates, predictions_kw["Transformer"][bloc], color="tab:red", label="Transformer")
plt.plot(dates, predictions_kw["Persistance (h-1)"][bloc], color="tab:gray", alpha=0.6, linestyle=":", label="Persistance")
plt.title("Puissance active moyenne horaire — sept jours consécutifs du test")
plt.ylabel("kW")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Expérience complémentaire : fenêtre de 168 heures (une semaine)

Pour répondre à la question 2 avec des chiffres, on réentraîne le même modèle avec une fenêtre de **168 heures**, et on compare les deux modèles **sur les mêmes heures de test**.

In [ ]:
X168, y168, cibles168, masques168 = preparer_fenetres(168)
debut = time.time()
modele168, histo168 = entrainer_energie(X168, y168, masques168, fenetre=168)
print(f"\nEntraînement terminé en {time.time() - debut:.0f} s")

test168 = masques168["test"]
position24 = {c: k for k, c in enumerate(cibles24)}
idx24 = np.array([position24[c] for c in cibles168[test168]])
reel_c = en_kw(y168[test168])
comparaison = {
    "Transformer, fenêtre 24 h": en_kw(predire(modele24, X24[idx24])),
    "Transformer, fenêtre 168 h": en_kw(predire(modele168, X168[test168])),
    "Persistance (h-1)": en_kw(X168[test168][:, -1, :]),
    "Référence journalière (h-24)": en_kw(X168[test168][:, -24, :]),
    "Référence hebdomadaire (h-168)": en_kw(X168[test168][:, 0, :]),
}
print(f"\nHeures de test communes : {test168.sum()}\n")
print(pd.DataFrame({nom: dict(zip(["MAE (kW)", "RMSE (kW)"], mae_rmse(reel_c, p)))
                    for nom, p in comparaison.items()}).T.round(4).to_string())

## 4. Questions de réflexion

**Le Transformer améliore-t-il les deux références ?**
Voir le premier tableau : on compare sa MAE et sa RMSE à celles de la persistance et de la référence journalière sur les mêmes heures.
La persistance est une référence solide à une heure d'avance (la consommation change souvent peu d'une heure à l'autre), mais elle réagit toujours **avec une heure de retard** aux changements, ce que montre le graphique des sept jours.
La référence journalière est plus faible : les habitudes d'un foyer ne se répètent pas exactement d'un jour à l'autre.
Une grande partie de la consommation (four, machine à laver…) reste imprévisible à partir de la seule courbe passée : le gain possible d'un modèle est donc limité.

**Une fenêtre de 168 heures capture-t-elle mieux les habitudes hebdomadaires ?**
Le deuxième tableau compare les deux fenêtres sur les mêmes heures. Une semaine d'historique donne au modèle le même jour de la semaine précédente, utile pour les habitudes liées au jour (week-end, jours travaillés).
En contrepartie, la séquence est 7 fois plus longue, l'attention coûte $168^2/24^2 \approx 49$ fois plus, et le modèle doit pondérer beaucoup plus de valeurs bruitées avec le même nombre d'exemples : le gain n'est pas garanti et doit être vérifié sur la validation.

**Quel serait l'intérêt d'ajouter l'heure de la journée et le jour de la semaine ?**
Le modèle ne connaît que des positions relatives (« il y a 5 heures »), pas l'heure réelle. Ajouter l'heure (0–23) et le jour (lundi–dimanche), par exemple en encodage cyclique $\sin(2\pi h/24), \cos(2\pi h/24)$ ou en embeddings appris, lui dit s'il prédit 19 h un dimanche ou 4 h un mardi.
Il peut alors apprendre les pics du soir et la différence semaine / week-end **sans avoir besoin d'une longue fenêtre**.

**Pourquoi normaliser uniquement à partir de l'entraînement ?**
La moyenne et l'écart-type font partie du modèle : les calculer sur toute la série ferait entrer des informations de la validation et du test dans l'entraînement (**fuite de données**).
Les scores de test seraient trop optimistes, alors qu'en conditions réelles on ne connaît pas encore les valeurs futures.

**La cible étant une puissance moyenne en kW, comment prédire une énergie en kWh ?**
Énergie = puissance × durée : $E\ (\text{kWh}) = \bar P\ (\text{kW}) \times \Delta t\ (\text{h})$.
Pour des pas d'une heure, l'énergie de l'heure vaut numériquement la puissance moyenne, si l'heure est complète.
Il faut donc agréger par la **somme** des mesures minute $\sum P_{\text{min}} / 60$ (et non la moyenne), en exigeant des heures complètes ou en complétant les minutes manquantes, puis entraîner sur cette cible ; pour un jour ou une semaine, on additionne les énergies horaires.